
<div style="text-align: center; padding: 30px 10px;">

<h1 style="color:#ff7500; font-size: 24px; margin-bottom: 10px;">
МФТИ ФПМИ
</h1>

<h2 style="font-size: 30px; margin-top: 5px;">
Практикум Python - Основной Поток
</h2>

<hr style="width: 60%; border: 1px solid #10069f; margin: 25px auto;">

<h3 style="font-size: 36px;">
5. Исключения. Контекстные менеджеры.
</h3>

<p style="margin-top: 20px;">
<strong>Дата:</strong> 03-05 марта 2026 года<br>
</p>

<p style="margin-top: 25px;">
Данный ноутбук является частью серии семинаров по курсу  
<em>«Практикум Python»</em> и предназначен для учебных и образовательных целей.
</p>

</div>

Часто в программах что-то идёт не так. Если ничего не предпринимать, они ломаются.

In [12]:
def parse_record(record: str) -> dict[str, int]:
    """Convert a tab-separated key=value string into a dictionary."""
    pairs = (item.split('=') for item in record.strip().split('\t'))
    return {key: int(value) for key, value in pairs}

entries = [
    'product_id=101\tviews=20\torders=3',
    'product_id=102\tviews=25\torders=4',
    'product_id=103\tviews=\torders=2',  # empty views
]

for entry in entries:
    print(parse_record(entry))

{'product_id': 101, 'views': 20, 'orders': 3}
{'product_id': 102, 'views': 25, 'orders': 4}


ValueError: invalid literal for int() with base 10: ''

Примеры исключений:

In [13]:
print(1 / 0)

ZeroDivisionError: division by zero

In [14]:
open('nonexistent.file')

FileNotFoundError: [Errno 2] No such file or directory: 'nonexistent.file'

## Иерархия встроенных исключений

```
BaseException
 ├── BaseExceptionGroup
 ├── GeneratorExit
 ├── KeyboardInterrupt
 ├── SystemExit
 └── Exception
      ├── ArithmeticError
      │    ├── FloatingPointError
      │    ├── OverflowError
      │    └── ZeroDivisionError
      ├── AssertionError
      ├── AttributeError
      ├── BufferError
      ├── EOFError
      ├── ExceptionGroup [BaseExceptionGroup]
      ├── ImportError
      │    └── ModuleNotFoundError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── MemoryError
      ├── NameError
      │    └── UnboundLocalError
      ├── OSError
      │    ├── BlockingIOError
      │    ├── ChildProcessError
      │    ├── ConnectionError
      │    │    ├── BrokenPipeError
      │    │    ├── ConnectionAbortedError
      │    │    ├── ConnectionRefusedError
      │    │    └── ConnectionResetError
      │    ├── FileExistsError
      │    ├── FileNotFoundError
      │    ├── InterruptedError
      │    ├── IsADirectoryError
      │    ├── NotADirectoryError
      │    ├── PermissionError
      │    ├── ProcessLookupError
      │    └── TimeoutError
      ├── ReferenceError
      ├── RuntimeError
      │    ├── NotImplementedError
      │    └── RecursionError
      ├── StopAsyncIteration
      ├── StopIteration
      ├── SyntaxError
      │    └── IndentationError
      │         └── TabError
      ├── SystemError
      ├── TypeError
      ├── ValueError
      │    └── UnicodeError
      │         ├── UnicodeDecodeError
      │         ├── UnicodeEncodeError
      │         └── UnicodeTranslateError
      └── Warning
           ├── BytesWarning
           ├── DeprecationWarning
           ├── EncodingWarning
           ├── FutureWarning
           ├── ImportWarning
           ├── PendingDeprecationWarning
           ├── ResourceWarning
           ├── RuntimeWarning
           ├── SyntaxWarning
           ├── UnicodeWarning
           └── UserWarning
```

Обработка исключений: `try...except...except`

In [ ]:
path = 'missing_data.txt'

try:
    file_obj = open(path, 'r')
except FileNotFoundError:
    print(f'File {path!r} was not found')
except (TypeError, ValueError, OSError) as err:
    print('Demonstrating handling multiple exception types in one clause')
except Exception as err:  # the first matching except block is executed
    print(f'An error occurred while opening {path!r}: {err!r}')

An error occurred while opening 'missing_data.txt': ZeroDivisionError()


Обработка исключений: `try...except...else...finally`

In [ ]:
stream = None

try:
    stream = open("data.txt", "r")  # potentially unsafe operation
    result = 10 / 0
except TypeError as err:  # wrong exception type (scope failure)
    print(f'An issue occurred: {err!r}')
else:  # scope success
    print('Operation completed successfully')
finally:  # scope exit
    if stream is not None:
        stream.close()
    print('This message is always displayed')


Ловить BaseException - **плохо!**

Почему?

In [ ]:
import time

while True:
    try:
        time.sleep(2)
        print('I am alive')
    except:
        print('Lol still alive')

(KeyboardInterrupt, BaseException, object)

In [ ]:
try:
    do_dangerous()
except:  # catch everything, even KeyboardInterrupt
    pass

# ==============================================

try:
    do_dangerous()
except BaseException:  # same as above
    pass

Бросить исключение можно с помощью ключевого слова `raise`

In [20]:
x = -5
if x < 0:
    raise ValueError('Positive integer expected')

ValueError: Positive integer expected

"Бросать" можно только `BaseException` или его наследника:

In [21]:
raise 50

TypeError: exceptions must derive from BaseException

Также бывают цепочки исключений:

In [5]:
def ctr(shows, clicks):
    """Returns banner click-through rate"""
    try:
        try:
            return clicks / shows
        except ZeroDivisionError as e:
            raise ValueError('Bad banner') from e
    except ValueError as e:
        raise OSError from e
        
ctr(0, 1)

OSError: 

Можно создавать свои классы исключений, достаточно отнаследоваться от `Exception`. Хорошая практика — наследовать свои исключения от общего предка, чтобы их было удобнее ловить. 

In [22]:
try:
    raise ValueError
except:
    raise RuntimeError

RuntimeError: 

In [ ]:
class FarmError(Exception):
    pass

class WrongFieldError(FarmError):
    pass

class WrongCropError(FarmError):
    pass

raise WrongCropError

WrongCropError: 

Иногда бывает исключение при обработке исключения

In [23]:
try:
    1 / 0
except ZeroDivisionError:
    raise ValueError("bad")  # второе исключение во время обработки первого


ValueError: bad

In [24]:
from warnings import warn

warn('something not right')

/var/folders/vp/0v99fjwx2nlcysq5ns40bc_w0000gn/T/ipykernel_84747/103442238.py:3: UserWarning: something not right
  warn('something not right')


In [10]:
# raise RuntimeError vs raise RuntimeError()
# разницы никакой - интерпретатор вызовет сам конструктор при raise, но нельзя указать сообщение в первом случае

## Менеджеры контекста

Нам хочется уметь открывать файлы и НЕ забывать их закрывать.

Как гарантировать, что некоторое действие будет выполнено вне зависимости от того, произошло исключение или нет?

In [ ]:
def do_something_dangerous(fd):
    raise RuntimeError('Not today!')

fd = open('myfile.txt', 'w')
try:
    do_something_dangerous(fd)
finally:
    print('Closing file')
    fd.close()
    print('File closed')

Closing file
File closed


RuntimeError: Not today!

Менеджеры контекста предоставляют удобный способ провести инициализацию и гарантированную финализацию "контекста".

In [ ]:
r = aquire_resource()
try:
    use_resource(r)
finally:
    release_resource(r)

In [ ]:
with open('../test.txt') as r:
    print(r.read())

  ___
| kek |
  ===
   \
    \
      ^__^
      (oo)\_______
      (__)\       )\/\
          ||----w |
          ||     ||



Примеры менеджеров контекста: `open`

In [30]:
with open('filename.txt', 'w') as fd:
    fd.write("Hello")
# file is closed
fd.write("world")

ValueError: I/O operation on closed file.

Примеры менеджеров контекста: `tempfile`

In [33]:
import tempfile
from time import sleep

with tempfile.TemporaryDirectory() as tmp:
    print(tmp)
    sleep(30)
# tmp file is removed

/var/folders/vp/0v99fjwx2nlcysq5ns40bc_w0000gn/T/tmpfrawk2eq


Синтаксис выражения `with`

In [ ]:
# nested contexts
with open('file1.txt') as file1, open('file2.txt') as file2:
    do_something(f, s)
# since python 3.11
with (open('file1.txt'), open('file2.txt')) as (file1, file2):
    do_something(f, s)

In [ ]:
# same as above
with first() as f:
    with second as s():
        do_something(f, s)

In [ ]:
with third():  # <as NAME> part as optional
    do_something()

Менеджеры контекста — объекты, реализующие специальный протокол

In [ ]:
import typing as tp
from types import TracebackType

class MyContextManager:
    def __enter__(self) -> tp.Self: # with () as X
        # initialize context
        return self
    
    def __exit__(self,
                 exc_type: type[BaseException] | None,
                 exc_value: BaseException | None,
                 traceback: TracebackType | None) -> bool | None:
        # finalize context
        if exc_value is not None:
            return True  # return True from __exit__ to suppress the exception

Семантика

In [ ]:
with acquire_resource() as resource:
    use_resource(resource)

In [ ]:
manager = acquire_resource()
resource = manager.__enter__()
try:
    use_resource(resource)
finally:
    exc_type, exc_value, traceback = sys.exc_info()
    suppress = manager.__exit__(exc_type, exc_value, traceback)
    if exc_value is not None and not suppress:
        raise exc_value

In [ ]:
class MyOpen:
    def __init__(self, path: str, mode: str = 'r'):
        self.path = path
        self.mode = mode
        self.fd = None

    def __enter__(self):
        self.fd = open(self.path, self.mode)
        return self.fd          # это попадает в `as`

    def __exit__(self, exc_type, exc_value, traceback):
        if self.fd:
            self.fd.close()
        return False            # не подавляем исключения

# использование
with MyOpen('file.txt', 'w') as f:
    f.write('hello')


Полушуточный пример

In [ ]:
class Tag:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print('<{}>'.format(self.name))
        return self
    
    def __exit__(self, *args):
        print('</{}>'.format(self.name))
        
    def update(self, new_name):
        self.name = new_name

        
with Tag('table') as table:
    table.update("broken_table")
    with Tag('tr'):
        with Tag('td'):
            print('cell 1')
        with Tag('td'):
            print('cell 2')

<table>
<tr>
<td>
cell 1
</td>
<td>
cell 2
</td>
</tr>
</broken_table>


`contextlib.contextmanager` — удобный способ создавать менеджеры контекста

In [34]:
from contextlib import contextmanager

@contextmanager
def mycm():
    print('before') # это __enter__
    yield 42  # yep, it is a generator  # значение идёт в `as r`, пауза
    print('after')  # это __exit__
    
with mycm() as r:
    print(f'got {r}')
    
with mycm() as r:
    raise RuntimeError('Oops')
# 'after' is not printed!

before
got 42
after
before


RuntimeError: Oops

Но работать с `contextlib.contextmanager` надо аккуратно

In [ ]:
from contextlib import contextmanager

@contextmanager
def mycm():
    print('before')
    try:
        yield ValueError("kek")  # предположим, не смогли открыть файл, возвращаем исключение
    finally:
        print('after')

with mycm() as r:
    print(type(r), r)
    raise RuntimeError('Oops')

before
<class 'ValueError'> kek
after


RuntimeError: Oops